### Topic modeling (BERTopic) + Keyword Extraction(KeyBERT)

In [3]:
import pandas as pd
from bertopic import BERTopic
from keybert import KeyBERT
from sklearn.feature_extraction.text import CountVectorizer

In [4]:
pd.set_option("display.max_colwidth", 150)

#Load data
(print("Loading Dataset---"))
df = pd.read_csv("C:/Users/Priyanka/Datascience/NLP_projects/cleaned_data.csv")

# Selecting random 8000 reviews
sample = df.sample(n=8000, random_state=42).reset_index(drop=True)
docs = sample["clean_text"].tolist()

#________Topic modeling with BERTopic______________
print("Running BERTopic (this embeds + clusters 8000 reviews)")
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1,2))
topic_model = BERTopic(
    vectorizer_model = vectorizer_model,
    min_topic_size = 50,
    verbose = True)

topics, probs = topic_model.fit_transform(docs)
sample["topic"] = topics

#Inspect topic found
topic_info = topic_model.get_topic_info()
print("\n" + "=" *50)
print(f" BERTopic found {len(topic_info)-1} topics (excluding outlier grp -1)")
print("\n" + "=" *50)
print(topic_info.head(15)) # top 15 topics

print("\n Outliers reviews(topic -1)")
n_outliers = (sample["topic"]==-1).sum()
print(f" {n_outliers} / {len(sample)} reviews ({n_outliers/len(sample):.1%})")

#print top words for 5 biggest topics
print("\n Top 5 topics with their words:")
for topic_id in topic_info[topic_info["Topic"] !=-1]["Topic"].head(5):
    words = [w for w, _ in topic_model.get_topic(topic_id)]
    print(f"Topic{topic_id}:{' , '.join(words[:8])}")

Loading Dataset---


2026-08-25 10:24:58,185 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic (this embeds + clusters 8000 reviews)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

2026-08-25 10:32:43,971 - BERTopic - Embedding - Completed ✓
2026-08-25 10:32:43,973 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-25 10:33:55,150 - BERTopic - Dimensionality - Completed ✓
2026-08-25 10:33:55,175 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-25 10:33:56,505 - BERTopic - Cluster - Completed ✓
2026-08-25 10:33:56,518 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-25 10:33:58,992 - BERTopic - Representation - Completed ✓



 BERTopic found 26 topics (excluding outlier grp -1)

    Topic  Count                             Name  \
0      -1   1913       -1_good_taste_like_product   
1       0   1119              0_food_dog_cat_dogs   
2       1   1103           1_coffee_cup_cups_like   
3       2    604       2_tea_green_teas_green tea   
4       3    334         3_sauce_hot_rice_chicken   
5       4    309        4_sugar_soda_drink_energy   
6       5    250     5_product_price_order_amazon   
7       6    248            6_baby_snack_son_food   
8       7    210          7_chips_potato_chip_bag   
9       8    208  8_gluten_bread_gluten free_free   
10      9    187  9_cookies_cookie_chocolate_oreo   
11     10    186       10_bars_bar_chocolate_like   
12     11    155       11_candy_candies_jelly_box   
13     12    142  12_oil_hair_coconut oil_coconut   
14     13    130      13_cereal_cereals_sugar_box   

                                                                    Representation  \
0         

In [8]:
# Keyword extraction with KeyBERT(per-review keywords)
print("\nRunning KeyBERT on smaller sample(500 reviews)")
kw_model = KeyBERT()
kw_sample = sample.sample(n=500, random_state=42).reset_index(drop=True)

def extract_keywords(text):
    try:
        kws = kw_model.extract_keywords(text, keyphrase_ngram_range=(1,2), stop_words="english",top_n=3)
        return ",".join([k for k, _ in kws])
    except Exception:
        return ""

kw_sample["keywords"] = kw_sample["clean_text"].apply(extract_keywords)
print(kw_sample[["clean_text","keywords"]].head(5))
      


Running KeyBERT on smaller sample(500 reviews)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

                                                                                                                                              clean_text  \
0  i bought this product for the home because we have avid soda flavored candy lovers just to be clear this product is not worth the price listed her...   
1  my dogs think these are the greatest thing since a t bone steak they don t last very long but it provides about hours of chew time for my golden l...   
2  this is the best of british tea available i buy it all the time this tea is really good for those people who want a strong cup of tea that can tak...   
3  i tried the peanut butter variety of oreo fudge cremes and they are definitely delicious and rich when i first opened the pack i was a little disa...   
4  my son loves these melts they are easy to take with us and great to mix with other treats to give as a healthy snack during the day i secretly lik...   

                                    keywords  
0  candyland col

In [9]:
kw_sample.to_csv("C:/Users/Priyanka/Datascience/NLP_projects/keywords.csv" , index=False)
print("\n Saved sucessfully")


 Saved sucessfully


In [17]:
# The actual insight: Merge topics with sentiment
print("\n Merging topics with sentiment to find which topics negative")

topic_sentiment = (sample[sample["topic"]!= -1].groupby("topic")["sentiment"].value_counts(normalize=True).unstack(fill_value=0))
#attach topic size and top words for readable summary
topic_sentiment["size"] = sample[sample["topic"] != -1].groupby("topic").size()
topic_sentiment["top_words"] = topic_sentiment.index.map(
    lambda t: ", ".join([w for w, _ in topic_model.get_topic(t)][:5])
)

#Sort by highest negative % 
topic_sentiment_sorted = topic_sentiment.sort_values("negative" , ascending=False)
print("\n" + "="*50)
print("\n TOPICS RANKED BY NEGATIVE SENTIMENT")
print("\n" + "="*50)
print(topic_sentiment_sorted.head(10).round(3))
    


 Merging topics with sentiment to find which topics negative


 TOPICS RANKED BY NEGATIVE SENTIMENT

sentiment  negative  neutral  positive  size  \
topic                                          
17            0.240    0.083     0.677    96   
18            0.222    0.086     0.691    81   
6             0.199    0.056     0.745   216   
12            0.186    0.117     0.697   145   
13            0.177    0.071     0.752   141   
3             0.171    0.087     0.743   439   
0             0.151    0.064     0.784  1117   
9             0.151    0.065     0.785   186   
7             0.137    0.043     0.820   211   
21            0.136    0.045     0.818    66   

sentiment                                 top_words  
topic                                                
17              jerky, beef, beef jerky, meat, jack  
18             product, flavor, tastes, great, like  
6            product, price, order, amazon, service  
12                  candy, candies, like, gift, box

In [18]:
topic_sentiment_sorted.to_csv("topic_sentiment_insight.csv")
print("\n Saved sucessfully")



 Saved sucessfully


In [5]:
topic_model.save("bertopic_model", serialization="safetensors")
sample.to_csv("C:/Users/Priyanka/Datascience/NLP_projects/topics.csv", index=False)
print("\nSaved model to bertopic_model/ and results to topics.csv")


Saved model to bertopic_model/ and results to topics.csv
